In [ ]:
yeonu@gpusystem:/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result$ ls
B_cell_Wilcoxon_DEG_crcatlas.csv           Mini_10percent_Wilcoxon_DEG_crcatlas.csv  Schwann_cell_Wilcoxon_DEG_crcatlas.csv
Cancer_cell_Wilcoxon_DEG_crcatlas.csv      Myeloid_cell_Wilcoxon_DEG_crcatlas.csv    Stromal_cell_Wilcoxon_DEG_crcatlas.csv
Epithelial_cell_Wilcoxon_DEG_crcatlas.csv  Neutrophil_Wilcoxon_DEG_crcatlas.csv      T_cell_Wilcoxon_DEG_crcatlas.csv
ILC_Wilcoxon_DEG_crcatlas.csv              NK_Wilcoxon_DEG_crcatlas.csv
Mast_cell_Wilcoxon_DEG_crcatlas.csv        Plasma_cell_Wilcoxon_DEG_crcatlas.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. DEG_Wilcoxon 전처리 
# Wilcoxon_DEG : Wilcoxon_DEG_~.csv

df_wilcoxon = pd.read_csv('/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result/Cancer_cell_Wilcoxon_DEG_crcatlas.csv')
print(df_wilcoxon.shape)
print(df_wilcoxon.head())
df_wilcoxon = df_wilcoxon[["names", "logfoldchanges", "pvals", "pvals_adj"]]
df_wilcoxon.rename(columns={"names": "gene","pvals": "pval","pvals_adj": "Wilcoxon_adj_pval"}, inplace=True)
print(df_wilcoxon.head())

(28476, 5)
     names     scores  logfoldchanges  pvals  pvals_adj
0   MT-CO1  214.45789        1.752644    0.0        0.0
1  RPL22L1  202.59563        2.039501    0.0        0.0
2   MT-ND5  187.28181        1.217037    0.0        0.0
3     AGR2  175.35300        1.232816    0.0        0.0
4   MT-CYB  175.17253        1.333463    0.0        0.0
      gene  logfoldchanges  pval  Wilcoxon_adj_pval
0   MT-CO1        1.752644   0.0                0.0
1  RPL22L1        2.039501   0.0                0.0
2   MT-ND5        1.217037   0.0                0.0
3     AGR2        1.232816   0.0                0.0
4   MT-CYB        1.333463   0.0                0.0


In [3]:
df_wilcoxon.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,MT-CO1,1.752644,0.0,0.0
1,RPL22L1,2.039501,0.0,0.0


## <font color="red">**데이터프레임**

In [4]:
import pandas as pd
from functools import reduce

# 1) 미리 각 df에서 gene + adj_pval 컬럼만 추출
dfs = [
    df_wilcoxon[["gene", "logfoldchanges", "pval", "Wilcoxon_adj_pval"]],
    # df_limmatrend[["gene", "limmatrend_adj_pval"]],
    # df_RISCQP[["gene", "RISCQP_adj_pval"]],
    # df_MAST[["gene", "MAST_adj_pval"]],
    # df_monocle_filtered[["gene", "monocle_adj_pval"]],
]

# 2) Reduce를 이용해 순차적으로 outer merge
df_mmm = reduce(
    lambda left, right: pd.merge(left, right, on="gene", how="outer"),
    dfs
)

# 결과 확인
print(df_mmm.shape)
print(df_mmm.head())

(28476, 4)
      gene  logfoldchanges  pval  Wilcoxon_adj_pval
0   MT-CO1        1.752644   0.0                0.0
1  RPL22L1        2.039501   0.0                0.0
2   MT-ND5        1.217037   0.0                0.0
3     AGR2        1.232816   0.0                0.0
4   MT-CYB        1.333463   0.0                0.0


In [5]:
df_mmm.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,MT-CO1,1.752644,0.0,0.0
1,RPL22L1,2.039501,0.0,0.0


In [6]:
# 1) 각 컬럼별 rank 계산 (값이 작을수록 1위, NaN은 가장 마지막)
df_mmm['rank_wilcoxon'] = df_mmm['Wilcoxon_adj_pval'] \
    .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_limma'] = df_mmm['limmatrend_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_RISCQP'] = df_mmm['RISCQP_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_MAST'] = df_mmm['MAST_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)


# 2) 5개 rank의 합 계산
df_mmm['rank_sum'] = df_mmm[
    ['rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
     ]
].sum(axis=1)

# 3) rank_sum 기준으로 정렬
df_mmm = df_mmm.sort_values('rank_sum').reset_index(drop=True)

# 결과 확인 (선택사항)
df_mmm

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
0,PCIF1,-0.690157,0.000000,0.0,1,1
1,NACA,-0.175865,0.000000,0.0,1,1
2,FABP6,-1.189613,0.000000,0.0,1,1
3,ENSG00000260342,-10.200455,0.000000,0.0,1,1
4,TRAF5,-1.076882,0.000000,0.0,1,1
...,...,...,...,...,...,...
28471,GGTLC1,-1.043872,0.990111,1.0,18427,18427
28472,TRGV5P,0.602284,0.990111,1.0,18427,18427
28473,KRTAP4-11,-0.283948,0.990072,1.0,18427,18427
28474,ENSG00000226904,-1.879814,0.990072,1.0,18427,18427


In [7]:
# df_mmm_re = df_mmm[['rank_sum', 'gene', 'rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
#                     ]]
# df_mmm_re.head(20)

In [8]:
# 원하는 순서의 리스트
genes_of_interest = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1", "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_mmm_filtered = df_mmm[df_mmm["gene"].isin(genes_of_interest)]
df_mmm_filtered

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
2669,FN1,-1.209574,2.594099e-155,2.766650e-154,2670,2670
10200,SCG5,1.039554,4.491339e-08,1.253753e-07,10201,10201
12408,IL11,-1.061487,1.934154e-03,4.438469e-03,12409,12409
14393,SDC1,-0.100520,7.550019e-02,1.493639e-01,14394,14394
17803,CLEC5A,0.059394,5.566845e-01,8.903700e-01,17804,17804
24044,CX3CR1,0.268019,9.927592e-01,1.000000e+00,18427,18427
24171,LYVE1,-0.555258,9.895043e-01,1.000000e+00,18427,18427
26367,NRAP,-0.175328,7.358254e-01,1.000000e+00,18427,18427
26527,VSIG4,0.000874,7.013821e-01,1.000000e+00,18427,18427


In [9]:
import pandas as pd
from functools import reduce

# 1️⃣ 각 메서드별 “gene + sign” DataFrame 생성
sign_dfs = []

# Wilcoxon: logfoldchanges >0 → MSI, <0 → MSS
sign_dfs.append(
    df_mmm_filtered[['gene','logfoldchanges']]
    .assign(Wilcoxon_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
    [['gene','Wilcoxon_sign']]
)

# limmatrend: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_limmatrend[['gene','logfoldchanges']]
#     .assign(limmat_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','limmat_sign']]
# )

# RISCQP: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_RISCQP[['gene','log2FC']]
#     .assign(RISCQP_sign=lambda d: d['log2FC'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','RISCQP_sign']]
# )

# monocle: log2FC >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_monocle[['gene','log2FC']]
#     .assign(monocle_sign=lambda d: d['log2FC'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','monocle_sign']]
# )

# # MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# 2️⃣ 모든 sign DataFrame outer merge
sign_df = reduce(lambda a,b: pd.merge(a,b,on='gene',how='outer'), sign_dfs)

# 3️⃣ rank 합계 df와 결합
df_final = df_mmm_filtered.merge(sign_df, on='gene', how='left')

# 4️⃣ consensus: MSI_sign이 3개 이상 → MSI_high, else MSS_high
# sign_cols = [col for col in df_final.columns if col.endswith('_sign')]
# df_final['consensus'] = df_final[sign_cols].apply(
#     lambda row: 'MSI_high' if (row=='MSI').sum() >= 3 else 'MSS_high',
#     axis=1
# )

# 5️⃣ 결과 확인 (상위 20개)
print(df_final.sort_values('rank_sum').head(20))

     gene  logfoldchanges           pval  Wilcoxon_adj_pval  rank_wilcoxon  \
0     FN1       -1.209574  2.594099e-155      2.766650e-154           2670   
1    SCG5        1.039554   4.491339e-08       1.253753e-07          10201   
2    IL11       -1.061487   1.934154e-03       4.438469e-03          12409   
3    SDC1       -0.100520   7.550019e-02       1.493639e-01          14394   
4  CLEC5A        0.059394   5.566845e-01       8.903700e-01          17804   
5  CX3CR1        0.268019   9.927592e-01       1.000000e+00          18427   
6   LYVE1       -0.555258   9.895043e-01       1.000000e+00          18427   
7    NRAP       -0.175328   7.358254e-01       1.000000e+00          18427   
8   VSIG4        0.000874   7.013821e-01       1.000000e+00          18427   

   rank_sum Wilcoxon_sign  
0      2670           MSS  
1     10201           MSI  
2     12409           MSS  
3     14394           MSS  
4     17804           MSI  
5     18427           MSI  
6     18427           M

In [10]:
df_final

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum,Wilcoxon_sign
0,FN1,-1.209574,2.594099e-155,2.766650e-154,2670,2670,MSS
1,SCG5,1.039554,4.491339e-08,1.253753e-07,10201,10201,MSI
2,IL11,-1.061487,1.934154e-03,4.438469e-03,12409,12409,MSS
3,SDC1,-0.100520,7.550019e-02,1.493639e-01,14394,14394,MSS
4,CLEC5A,0.059394,5.566845e-01,8.903700e-01,17804,17804,MSI
5,CX3CR1,0.268019,9.927592e-01,1.000000e+00,18427,18427,MSI
6,LYVE1,-0.555258,9.895043e-01,1.000000e+00,18427,18427,MSS
7,NRAP,-0.175328,7.358254e-01,1.000000e+00,18427,18427,MSS
8,VSIG4,0.000874,7.013821e-01,1.000000e+00,18427,18427,MSI


In [11]:
import numpy as np

# 1) 기존 neg_log10_adj_pval 계산
epsilon = np.nextafter(0, 1)
df_final['neg_log10_adj_pval'] = -np.log10(
    df_final['Wilcoxon_adj_pval'].replace(0, epsilon)
)

# 2) 지수표현(예: 2.44e+02)으로 소수점 둘째 자리까지 포맷
df_final['neg_log10_adj_pval'] = df_final['neg_log10_adj_pval'] \
    .apply(lambda x: f"{x:.1e}")

# 3) gene 순서 재배열
desired_order = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1",
                 "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_final = (
    df_final
    .set_index('gene')
    .reindex(desired_order)
    .reset_index()
)

# 확인
print(df_final[['gene', 'neg_log10_adj_pval', 'Wilcoxon_sign']])

     gene neg_log10_adj_pval Wilcoxon_sign
0   VSIG4           -0.0e+00           MSI
1   LYVE1           -0.0e+00           MSS
2  CLEC5A            5.0e-02           MSI
3  CX3CR1           -0.0e+00           MSI
4     FN1            1.5e+02           MSS
5    IL11            2.4e+00           MSS
6    NRAP           -0.0e+00           MSS
7    SDC1            8.3e-01           MSS
8    SCG5            6.9e+00           MSI
